# S3 - Procesamiento y calidad de datos: filtrado, duplicados, nulos y particionamiento analitico

**Actividad:** construir el notebook `03_procesamiento_calidad_datos_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), validando esquema, filtrando y ordenando resultados, tratando duplicados y nulos, y escribiendo una salida analitica particionada en Parquet — sobre el dataset real H&M ya usado en S2 (`customers.csv`, `articles.csv`).

**Proposito de la actividad:** dejar evidencia ejecutable de que dominas los controles de calidad de datos (esquema, filtrado, orden, duplicados, nulos) y el particionamiento de salidas analiticas — antes de avanzar a ML distribuido (S4).

Guia completa: `docs/sesiones/S03_Procesamiento_Calidad_Datos_Particionamiento.md`, seccion 3.

## 3.1 Preparar los datos de S3 y reanudar el entorno `lambda26`

**Producto del paso:** `customers.csv` y `articles.csv` disponibles en `pyspark/sesiones/s03-procesamiento-calidad-datos/data/`, entorno `lambda26` funcionando.

Ya descargaste estos dos archivos en S2 — no hace falta descargarlos de nuevo, solo copialos a la carpeta de esta sesion (desde tu maquina, no dentro del notebook):

```bash
cp lambda26/pyspark/sesiones/s02-fundamentos/data/customers.csv lambda26/pyspark/sesiones/s03-procesamiento-calidad-datos/data/
cp lambda26/pyspark/sesiones/s02-fundamentos/data/articles.csv lambda26/pyspark/sesiones/s03-procesamiento-calidad-datos/data/
```

`customers.csv` pesa ~207 MB — la copia tarda unos segundos, no es instantanea. **Espera a que termine antes de abrir Jupyter y correr el notebook**: si lees el archivo mientras todavia se esta copiando, Spark lee la foto parcial que existe en ese instante, sin ningun error. Confirma que la copia termino:

```bash
wc -l lambda26/pyspark/sesiones/s02-fundamentos/data/customers.csv
wc -l lambda26/pyspark/sesiones/s03-procesamiento-calidad-datos/data/customers.csv
```

Esta sesion no necesita `transactions.parquet` — el foco es esquema, filtrado, orden, duplicados y nulos sobre datos tabulares, no sobre transacciones.

## 3.2 Crear el notebook y la `SparkSession`

**Producto del paso:** notebook con una `SparkSession` activa.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion3-calidad-datos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/02 19:22:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/02 19:22:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/02 19:22:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


`spark.driver.memory` en `4g` desde el arranque — en S2 la JVM se cayo por quedarse en el default de 1g; aca se fija de una vez.

In [2]:
ORIGEN_DATOS = "/opt/s03-procesamiento-calidad-datos/data"
ARTIFACTS = "/opt/s03-procesamiento-calidad-datos/artifacts"

## 3.3 Cargar `customers.csv` y validar el esquema

**Producto del paso:** `df_customers` cargado con esquema explicito, verificado contra lo esperado — control de calidad #1: esquema.

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema_customers = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("FN", DoubleType(), nullable=True),
    StructField("Active", DoubleType(), nullable=True),
    StructField("club_member_status", StringType(), nullable=True),
    StructField("fashion_news_frequency", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("postal_code", StringType(), nullable=True),
])

df_customers = spark.read.csv(
    f"{ORIGEN_DATOS}/customers.csv",
    header=True,
    schema=schema_customers,
)

df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- club_member_status: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



Un esquema explicito evita que el tipo cambie entre corridas, pero no confirma por si solo que el archivo real tenga las columnas que esperabas — un CSV con una columna renombrada o faltante igual carga, sin error, con esa columna llena de `null`. Valida la presencia de las columnas requeridas de forma explicita, antes de seguir:

In [4]:
columnas_requeridas = {
    "customer_id",
    "FN",
    "Active",
    "club_member_status",
    "fashion_news_frequency",
    "age",
    "postal_code",
}

faltantes = columnas_requeridas - set(df_customers.columns)
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias: {sorted(faltantes)}")

print("Esquema validado: las 7 columnas requeridas estan presentes.")

Esquema validado: las 7 columnas requeridas estan presentes.


Confirma tambien el conteo de filas — 7 columnas, en el mismo orden y tipo:

In [5]:
print(df_customers.columns)
df_customers.count()

['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']


1371980

## 3.4 Explorar nulos por columna

**Producto del paso:** conteo exacto de nulos por columna, con porcentaje sobre el total.

In [6]:
from pyspark.sql.functions import col, count, when

total_filas = df_customers.count()

df_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns
]).show(vertical=True, truncate=False)

[Stage 6:====>                                                    (1 + 11) / 12]

-RECORD 0------------------------
 customer_id            | 0      
 FN                     | 895050 
 Active                 | 907576 
 club_member_status     | 6062   
 fashion_news_frequency | 16009  
 age                    | 15861  
 postal_code            | 0      



El mismo resultado, con porcentaje:

In [7]:
nulos = df_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns
]).collect()[0].asDict()

for columna, cantidad in nulos.items():
    porcentaje = cantidad / total_filas * 100
    print(f"{columna}: {cantidad} nulos ({porcentaje:.1f}%)")

[Stage 9:====>                                                    (1 + 11) / 12]

customer_id: 0 nulos (0.0%)
FN: 895050 nulos (65.2%)
Active: 907576 nulos (66.2%)
club_member_status: 6062 nulos (0.4%)
fashion_news_frequency: 16009 nulos (1.2%)
age: 15861 nulos (1.2%)
postal_code: 0 nulos (0.0%)


`isNull()` no detecta todo lo que en la practica significa "sin dato": una columna de texto puede traer una cadena vacia (`""`) o solo espacios, y ninguna de las dos cuenta como `NULL` para Spark. Antes de dar por buena una columna de texto sin nulos, confirma tambien esto (aca sobre `customer_id`, la columna critica del dataset):

In [8]:
from pyspark.sql.functions import trim

df_customers.filter(
    col("customer_id").isNull() | (trim(col("customer_id")) == "")
).count()

0

## 3.5 Filtrado de datos (`filter()`/`where()`)

**Producto del paso:** las dos sintaxis de `filter()` (SQL y booleana) aplicadas sobre datos reales, mas filtrado de nulos y de contenido de texto.

Antes de escribir un filtro compuesto: los operadores de Python `and`, `or` y `not` **no** funcionan sobre columnas de Spark (fallan con un error, no dan un resultado silenciosamente incorrecto) — se usan `&`, `|` y `~`, con cada comparacion entre parentesis. Todos los ejemplos de esta seccion ya siguen esa regla.

Expresion SQL como texto:

In [9]:
df_customers.filter("age > 30").show(3)
df_customers.filter("club_member_status = 'ACTIVE'").show()
df_customers.filter("age BETWEEN 25 AND 35").show()
df_customers.filter("fashion_news_frequency IN ('Regularly', 'Monthly')").show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows
+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+-----

La misma logica, con `col()` — `.between(25, 35)` es el equivalente exacto de `BETWEEN 25 AND 35`, **incluye ambos extremos**:

In [10]:
from pyspark.sql.functions import col

df_customers.filter(col("age") > 30).show()
df_customers.filter(col("club_member_status") == "ACTIVE").show()
df_customers.filter(col("age").between(25, 35)).show()
df_customers.filter(col("fashion_news_frequency").isin("Regularly", "Monthly")).show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|00007d2de826758b6...| 1.0|   1.0|            ACTIVE|             Regularly| 32|8d6f45050876d059c...|
|000097d91384a0c14...|NULL|  NULL|            ACTIVE|                  NONE| 31|2c29ae653a9282cce...|
|00009c2aeae8761f7...|NULL|  NULL|            ACTIVE|                  NONE| 49|7e2caa18837edc6a7...|
|00009d946eec3ea54...| 1.0|   1.0|            ACTIVE|             Regularly| 56|b3

Cuidado con la trampa: `(col("age") > 25) & (col("age") < 35)` **no** es lo mismo que `BETWEEN 25 AND 35` ni que `.between(25, 35)` — esa version excluye ambos extremos (25 y 35 quedan fuera). Compara las tres formas para confirmarlo:

In [11]:
print("between (incluye 25 y 35):", df_customers.filter(col("age").between(25, 35)).count())
print("estricto (excluye 25 y 35):   ", df_customers.filter((col("age") > 25) & (col("age") < 35)).count())

between (incluye 25 y 35): 413179


[Stage 26:>                                                       (0 + 12) / 12]

estricto (excluye 25 y 35):    338303


`eqNullSafe()` compara igualdad tratando `null` como un valor comparable, no como "desconocido" — `col("x") == None` nunca es verdadero (ni siquiera para filas con `x` nulo), pero `col("x").eqNullSafe(None)` si encuentra esas filas:

In [12]:
df_customers.filter(col("club_member_status").eqNullSafe("ACTIVE")).show(3)
df_customers.filter(col("club_member_status").eqNullSafe(None)).count()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


6062

`where()` es el mismo metodo que `filter()`, con otro nombre:

In [13]:
df_customers.where(col("Active") == 1).show()
df_customers.where("FN = 1").show()

+--------------------+---+------+------------------+----------------------+---+--------------------+
|         customer_id| FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+---+------+------------------+----------------------+---+--------------------+
|00006413d8573cd20...|1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|00007d2de826758b6...|1.0|   1.0|            ACTIVE|             Regularly| 32|8d6f45050876d059c...|
|00009d946eec3ea54...|1.0|   1.0|            ACTIVE|             Regularly| 56|b31984b20a8c478de...|
|0000ae1bbb25e04bd...|1.0|   1.0|            ACTIVE|             Regularly| 29|2c29ae653a9282cce...|
|0000b2f1829e23b24...|1.0|   1.0|            ACTIVE|             Regularly| 54|ca8ca81e8b5794992...|
|0000b95f630aaa931...|1.0|   1.0|            ACTIVE|             Regularly| 49|f63abf76506122d9f...|
|0000d6c053fc8f938...|1.0|   1.0|            ACTIVE|             Regularly| 41|5b5f53c673d0

Filtrar nulos de una columna especifica — a diferencia del conteo de 3.4, esto deja *ver* las filas, no solo contarlas:

In [14]:
df_customers.filter(col("club_member_status").isNotNull()).show()
df_customers.filter(col("fashion_news_frequency").isNull()).show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|0000757967448a6cb...|NULL|  NULL|            ACTIVE|                  NONE| 20|fe7b8e2b3fafb89ca...|
|00007d2de826758b6...| 1.0|   1.0|            ACTIVE|             Regularly| 32|8d

Filtrar por contenido de texto — util para validar formato:

In [15]:
df_customers.filter(col("postal_code").startswith("28")).show()
df_customers.filter(col("postal_code").contains("56")).show()
df_customers.filter(col("postal_code").endswith("00")).show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00068c36f7034bf3e...|NULL|  NULL|        PRE-CREATE|                  NONE| 20|28f3300ab2f2a41b2...|
|0016dc6d4cbacb34a...|NULL|  NULL|            ACTIVE|                  NONE| 23|2843f92d2714b40b2...|
|0036620fa407ca305...|NULL|  NULL|            ACTIVE|                  NONE| 22|288e4199ca26dfb67...|
|0037c4ae2ea5c547b...| 1.0|   1.0|            ACTIVE|             Regularly| 21|28f554cac7b568926...|
|00491b4e99e2d274c...| 1.0|   1.0|            ACTIVE|             Regularly| 51|280ec823d1a983daa...|
|004b239de0829da59...|NULL|  NULL|            ACTIVE|                  NONE| 41|285780b21ae65addb...|
|0050b9f0b14dfc5bd...|NULL|  NULL|            ACTIVE|                  NONE| 27|28

Filtrar tambien sirve para validar rangos — confirma si hay edades fuera de lo razonable:

In [16]:
df_edad_invalida = df_customers.filter((col("age") < 0) | (col("age") > 100))
df_edad_invalida.count()

0

Si el conteo da 0, tambien es un control de calidad exitoso — no un resultado "vacio" sin valor.

## 3.6 Ordenar resultados (`orderBy()`/`sort()`)

**Producto del paso:** resultados ordenados por una, varias y por una expresion sobre una columna.

`orderBy()` reorganiza **todo** el DataFrame en un orden global — a diferencia de un filtro, que cada particion resuelve por su cuenta, ordenar de punta a punta obliga a Spark a barajar (*shuffle*) los datos entre particiones para compararlos entre si. Es una operacion cara: aplicala sobre el resultado final que vas a mostrar o guardar, no como paso intermedio de un pipeline si el orden no es necesario ahi.

Por una sola columna, en formas equivalentes:

In [17]:
df_customers.orderBy("age").show(3)
df_customers.orderBy(col("age")).show(3)
df_customers.orderBy(col("age").desc()).show(3)
df_customers.orderBy("age", ascending=False).show(3)

+--------------------+----+------+------------------+----------------------+----+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency| age|         postal_code|
+--------------------+----+------+------------------+----------------------+----+--------------------+
|ef8b1d93a9513b3f2...|NULL|  NULL|            ACTIVE|                  NONE|NULL|e3bf34653a9245100...|
|ef95690cebbfedf0d...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|472817c20f359aef5...|
|ef902e65aa18db0b5...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|45e4bc6b36c1ca02a...|
+--------------------+----+------+------------------+----------------------+----+--------------------+
only showing top 3 rows


+--------------------+----+------+------------------+----------------------+----+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency| age|         postal_code|
+--------------------+----+------+------------------+----------------------+----+--------------------+
|ef8b1d93a9513b3f2...|NULL|  NULL|            ACTIVE|                  NONE|NULL|e3bf34653a9245100...|
|ef95690cebbfedf0d...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|472817c20f359aef5...|
|ef902e65aa18db0b5...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|45e4bc6b36c1ca02a...|
+--------------------+----+------+------------------+----------------------+----+--------------------+
only showing top 3 rows


+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|e8b2a7bf44f42e808...|NULL|  NULL|            ACTIVE|                  NONE| 99|cc02109b1630ec723...|
|eeaeb36eecef27871...| 1.0|   1.0|            ACTIVE|             Regularly| 99|2c29ae653a9282cce...|
|7558adbc0401acdd7...|NULL|  NULL|            ACTIVE|                  NONE| 99|f85c74e22df88aabb...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


[Stage 46:=========>                                              (2 + 10) / 12]

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|e8b2a7bf44f42e808...|NULL|  NULL|            ACTIVE|                  NONE| 99|cc02109b1630ec723...|
|eeaeb36eecef27871...| 1.0|   1.0|            ACTIVE|             Regularly| 99|2c29ae653a9282cce...|
|687c675e64e5f5d96...|NULL|  NULL|        PRE-CREATE|                  NONE| 99|7d2ef4ec3ff9ea93c...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


Por defecto, los nulos van al final en orden ascendente y al principio en descendente — `asc_nulls_last()`/`desc_nulls_last()` (o sus pares `_nulls_first()`) lo dejan explicito en vez de depender del comportamiento por defecto:

In [18]:
df_customers.orderBy(col("age").desc_nulls_last()).show(3)
df_customers.orderBy(col("postal_code").asc_nulls_last()).show(3)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|e8b2a7bf44f42e808...|NULL|  NULL|            ACTIVE|                  NONE| 99|cc02109b1630ec723...|
|eeaeb36eecef27871...| 1.0|   1.0|            ACTIVE|             Regularly| 99|2c29ae653a9282cce...|
|687c675e64e5f5d96...|NULL|  NULL|        PRE-CREATE|                  NONE| 99|7d2ef4ec3ff9ea93c...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


[Stage 48:>                                                       (0 + 12) / 12]

+--------------------+---+------+------------------+----------------------+---+--------------------+
|         customer_id| FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+---+------+------------------+----------------------+---+--------------------+
|36d581d9301434356...|1.0|   1.0|            ACTIVE|             Regularly| 52|0000198d2c593b7d3...|
|f9a373ae2dabf53c3...|1.0|   1.0|            ACTIVE|             Regularly| 31|00005652fb5323679...|
|078726bc6b22124b5...|1.0|   1.0|            ACTIVE|             Regularly| 30|00005652fb5323679...|
+--------------------+---+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


Por varias columnas, cada una con su propio sentido:

In [19]:
df_customers.orderBy(col("club_member_status").asc(), col("age").desc()).show(3)
df_customers.orderBy(["club_member_status", "age"], ascending=[True, False]).show(3)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|59027ba9ec753065f...|NULL|  NULL|              NULL|                  NONE| 93|32d5efd0e4ea5baf4...|
|eb65e726a334a1a72...|NULL|  NULL|              NULL|                  NONE| 92|98b4a470dfc22c224...|
|e8bdd5514aaee8211...|NULL|  NULL|              NULL|                  NONE| 86|16794f131214279e0...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


[Stage 50:>                                                       (0 + 12) / 12]

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|59027ba9ec753065f...|NULL|  NULL|              NULL|                  NONE| 93|32d5efd0e4ea5baf4...|
|eb65e726a334a1a72...|NULL|  NULL|              NULL|                  NONE| 92|98b4a470dfc22c224...|
|e8bdd5514aaee8211...|NULL|  NULL|              NULL|                  NONE| 86|16794f131214279e0...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


Por una expresion, no solo por el valor de la columna — aca, por la longitud del codigo postal:

In [20]:
from pyspark.sql.functions import length

df_customers.orderBy(length(col("postal_code")).desc()).show()

[Stage 51:====>                                                   (1 + 11) / 12]

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|ef7779b451400dc29...|NULL|  NULL|            ACTIVE|                  NONE| 55|a4cd4a9969502597d...|
|ef77880cfa82fed21...| 1.0|   1.0|            ACTIVE|             Regularly| 46|3d62fbf80d5e061d3...|
|ef77931ee8bb92fa1...|NULL|  NULL|            ACTIVE|                  NONE| 33|2c29ae653a9282cce...|
|ef77b2ed0371b9e10...| 1.0|   1.0|            ACTIVE|             Regularly| 67|25527ee2a1279e861...|
|ef77b42fa514b42d0...| 1.0|   1.0|            ACTIVE|             Regularly| 44|8c790320fea676232...|
|ef77ccd82a1aaf209...|NULL|  NULL|            ACTIVE|                  NONE| 30|6a1050c4847b10f54...|
|ef77d54694e2d4f98...|NULL|  NULL|            ACTIVE|                  NONE| 32|94

`sort()` es el mismo metodo que `orderBy()`, con otro nombre:

In [21]:
df_customers.sort("age").show(3)
df_customers.sort(col("age").desc()).show(3)

+--------------------+----+------+------------------+----------------------+----+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency| age|         postal_code|
+--------------------+----+------+------------------+----------------------+----+--------------------+
|ef8b1d93a9513b3f2...|NULL|  NULL|            ACTIVE|                  NONE|NULL|e3bf34653a9245100...|
|ef95690cebbfedf0d...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|472817c20f359aef5...|
|ef902e65aa18db0b5...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|45e4bc6b36c1ca02a...|
+--------------------+----+------+------------------+----------------------+----+--------------------+
only showing top 3 rows


[Stage 53:====>                                                   (1 + 11) / 12]

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|687c675e64e5f5d96...|NULL|  NULL|        PRE-CREATE|                  NONE| 99|7d2ef4ec3ff9ea93c...|
|3b9ec1854ba779c3c...| 1.0|   1.0|            ACTIVE|             Regularly| 99|2c29ae653a9282cce...|
|3b28156770e0a1743...|NULL|  NULL|            ACTIVE|                  NONE| 99|1cbcd2b9e62968946...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


## 3.7 Tratamiento de duplicados

**Producto del paso:** duplicados identificados y/o tratados con varias tecnicas — control de calidad #2.

Antes de eliminar nada, identifica **sin eliminar todavia** -- lista que valores de `customer_id` aparecen mas de una vez, para saber si hay algo que tratar antes de decidir como tratarlo:


In [22]:
from pyspark.sql.functions import count

df_customers.groupBy("customer_id").count().filter("count > 1").show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



Recien ahora, con el diagnostico en mano, elimina duplicados completos o por columnas especificas:


In [23]:
df_clean = df_customers.dropDuplicates()
df_clean = df_customers.dropDuplicates(["customer_id"])
df_clean = df_customers.dropDuplicates(["customer_id", "postal_code"])

`distinct()` es la forma corta de `dropDuplicates()` sin argumentos:

In [24]:
df_clean = df_customers.distinct()

Confirma las cifras sobre el dataset completo — si coinciden con el total, no hay duplicados reales:

In [25]:
total = df_customers.count()
sin_duplicados_fila_completa = df_customers.distinct().count()
sin_duplicados_por_id = df_customers.dropDuplicates(["customer_id"]).count()

print(f"Total: {total}, sin duplicar (fila completa): {sin_duplicados_fila_completa}, sin duplicar (por customer_id): {sin_duplicados_por_id}")

[Stage 72:=======>                                                  (1 + 7) / 8]

Total: 1371980, sin duplicar (fila completa): 1371980, sin duplicar (por customer_id): 1371980


En una corrida real, las tres cifras dieron **1 371 980** — cero duplicados, en ninguna de las dos definiciones.

Marcar duplicados eligiendo cual fila conservar (aca, la de mayor `age` por `customer_id`). Con un solo criterio de orden, si dos filas del mismo cliente empatan en `age`, cual de las dos "gana" el `row_number() == 1` es arbitrario — no determinista entre corridas. Se agrega `postal_code` como desempate, para que el resultado sea siempre el mismo dado el mismo dataset:

In [26]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("customer_id").orderBy(
    col("age").desc_nulls_last(),
    col("postal_code").asc_nulls_last(),
)

df_ranked = df_customers.withColumn("row_num", row_number().over(window_spec))
df_clean = df_ranked.filter(col("row_num") == 1).drop("row_num")

**Contraste real con `articles.csv`** (S2, 3.10): `rdd.take(5)` sobre `detail_desc` trajo descripciones identicas repetidas. Eso **no** son duplicados de fila — cada `article_id` es distinto (variante de color/talla). Confirma cuantos `article_id` comparten la misma descripcion, sin tratarlos como error:

In [27]:
df_articles = spark.read.csv(f"{ORIGEN_DATOS}/articles.csv", header=True, inferSchema=True)

duplicados_por_descripcion = (
    df_articles.filter(col("detail_desc").isNotNull())
    .groupBy("detail_desc")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)
duplicados_por_descripcion.show(5, truncate=False)

[Stage 78:======>                                                   (1 + 8) / 9]

+---------------------------------------------------------------------+-----+
|detail_desc                                                          |count|
+---------------------------------------------------------------------+-----+
|T-shirt in printed cotton jersey.                                    |159  |
|Leggings in soft organic cotton jersey with an elasticated waist.    |138  |
|T-shirt in soft, printed cotton jersey.                              |137  |
|Socks in a soft, jacquard-knit cotton blend with elasticated tops.   |136  |
|Fine-knit trainer socks in a soft cotton blend with elasticated tops.|134  |
+---------------------------------------------------------------------+-----+
only showing top 5 rows


`filter(col("detail_desc").isNotNull())` va **antes** del `groupBy()`: sin el, agrupa todos los articulos sin descripcion bajo un mismo grupo `NULL` — que en una corrida real salio como el "valor mas repetido" (416 articulos), tapando los duplicados de contenido real.

Aca si corresponde `Window`+`row_number()` para elegir un representante por `product_code` (el producto base, sin variantes) — `article_id` identifica cada variante, `product_code` el producto. `article_id` ya es unico por fila, asi que ordenar por el alcanza como desempate:

In [28]:
window_producto = Window.partitionBy("product_code").orderBy("article_id")

df_articles_un_por_producto = (
    df_articles
    .withColumn("fila", row_number().over(window_producto))
    .filter(col("fila") == 1)
    .drop("fila")
)

print(f"Filas originales: {df_articles.count()}, un representante por product_code: {df_articles_un_por_producto.count()}")

[Stage 84:======>                                                   (1 + 8) / 9]

Filas originales: 105542, un representante por product_code: 47224


En una corrida real, la reduccion fue de **105 542 filas a 47 224 representantes** — mas de la mitad de `articles.csv` son variantes de un producto ya representado por otra fila.

## 3.8 Tratar nulos con `.na.fill()` y `.na.drop()`

**Producto del paso:** `df_customers_valido`, el dataset final — con nulos tratados columna por columna, cada decision con un criterio documentado — control de calidad #3.

`FN`/`Active` son columnas de tipo "bandera"; un nulo ahi significa "la bandera no se activo" — se rellenan con `0`. `fillna()` es un alias exacto de `.na.fill()`:

In [29]:
df_fill2 = df_customers.fillna({"FN": 0, "Active": 0})

`.na.fill()` puede rellenar cualquier columna, con cualquier tipo de valor — incluida `age`, con `0`. Pero que la sintaxis lo permita no lo hace buena idea: un cliente de "0 anos" es un dato **falso** que se ve como valido, peor que dejarlo nulo. Por eso la version que este notebook aplica de verdad **no** rellena `age`:

In [30]:
df_customers_limpio = df_customers.na.fill({
    "FN": 0,
    "Active": 0,
    "fashion_news_frequency": "NONE",
    "club_member_status": "UNKNOWN",
})

`.na.drop()` sin argumentos elimina toda fila con **cualquier** nulo, en cualquier columna — sobre este dataset (FN/Active ~65% nulos), descartaria la enorme mayoria de las filas. Pruebalo para ver la magnitud, pero no lo uses como version final:

In [31]:
df_customers.na.drop().count()

462911

En una corrida real, dio **462 911** — de 1 371 980 filas, sobreviven menos de un tercio (33.7%). Compara esto contra el resultado de `na.drop(subset=["customer_id"])` de abajo, que no elimina ninguna: la diferencia entre "cualquier columna" y "solo la columna critica" no es un matiz, es la diferencia entre destruir dos tercios del dataset o no perder nada.

`customer_id` es la columna critica — corresponde `.na.drop(subset=[...])`, apuntando solo a esa columna:

In [32]:
df_customers_valido = df_customers_limpio.na.drop(subset=["customer_id"])

print(f"Filas antes: {df_customers.count()}, despues de na.drop(subset=['customer_id']): {df_customers_valido.count()}")

[Stage 96:====>                                                   (1 + 11) / 12]

Filas antes: 1371980, despues de na.drop(subset=['customer_id']): 1371980


En una corrida real, ambos numeros dieron **1 371 980** — `customer_id` nunca llega nulo en este dataset. Confirma tambien que la columna critica quedo realmente completa, no solo de un vistazo al conteo:

In [33]:
assert df_customers_valido.filter(col("customer_id").isNull()).count() == 0

`df_customers_valido` se reutiliza en los pasos que siguen (3.9-3.11), varios con su propio `.count()` — `cache()` guarda el resultado la primera vez que una accion lo dispara:

In [34]:
df_customers_valido = df_customers_valido.cache()

Si el DataFrame fuera mas grande de lo que la memoria disponible aguanta, `persist(StorageLevel.MEMORY_AND_DISK)` es la version con mas control — cae a disco en vez de fallar:

In [35]:
from pyspark.storagelevel import StorageLevel

df_customers_valido.persist(StorageLevel.MEMORY_AND_DISK)

26/09/02 19:23:43 WARN CacheManager: Asked to cache already cached data.


DataFrame[customer_id: string, FN: double, Active: double, club_member_status: string, fashion_news_frequency: string, age: int, postal_code: string]

## 3.9 Escritura en multiples formatos

**Producto del paso:** el mismo resultado guardado en tres formatos distintos, con `.write.format()` — sobre una muestra chica, no el dataset completo, solo para ver la sintaxis.

In [36]:
muestra = df_edad_invalida.limit(100)  # resultado (vacio o no) de la validacion de 3.5

muestra.write.format("csv").option("header", True).mode("overwrite").save(f"{ARTIFACTS}/muestra_csv")
muestra.write.format("json").mode("overwrite").save(f"{ARTIFACTS}/muestra_json")
muestra.write.format("parquet").mode("overwrite").save(f"{ARTIFACTS}/muestra_parquet")

## 3.10 Escritura particionada en Parquet

**Producto del paso:** salida analitica particionada por `club_member_status`, lista para BI/ML.

Una buena columna de particion tiene cardinalidad baja o moderada (pocos valores distintos) y aparece seguido en filtros — `club_member_status` cumple ambas. Un identificador unico como `customer_id` **no** serviria: crearia una carpeta distinta por cada uno de los 1 371 980 clientes, la mayoria con un solo archivo diminuto adentro — el particionamiento deja de ayudar y solo agrega miles de carpetas.

`repartition(4)` antes de escribir influye en cuantos archivos `part-0000X` caen dentro de cada carpeta de particion — no es una garantia exacta de "4 archivos por carpeta" (depende de como queden distribuidos los datos entre esas 4 particiones), pero evita heredar sin control el numero de particiones de la lectura original:

In [37]:
(
    df_customers_valido
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("club_member_status")
    .save(f"{ARTIFACTS}/customers_particionado")
)

`partitionBy("club_member_status")` crea una subcarpeta por cada valor distinto de esa columna. Verifica la estructura real:

In [38]:
import os

for carpeta in sorted(os.listdir(f"{ARTIFACTS}/customers_particionado")):
    print(carpeta)

._SUCCESS.crc
_SUCCESS
club_member_status=ACTIVE
club_member_status=LEFT CLUB
club_member_status=PRE-CREATE
club_member_status=UNKNOWN


Contraste directo: si en vez de `repartition(4)` usas `coalesce(1)`, obtenes un solo archivo por carpeta de particion:

In [39]:
(
    df_customers_valido
    .coalesce(1)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("club_member_status")
    .save(f"{ARTIFACTS}/customers_particionado_un_archivo")
)

## 3.11 Leer de vuelta y verificar el particionamiento

**Producto del paso:** confirmacion de que la salida particionada se lee correctamente y que el particionamiento si se aprovecha en consultas.

In [40]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/customers_particionado")
df_verificacion.printSchema()
df_verificacion.count()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- club_member_status: string (nullable = true)



1371980

El conteo de vuelta deberia coincidir exactamente con el dataset que escribiste — confirmalo en vez de asumirlo:

In [41]:
assert df_verificacion.count() == df_customers_valido.count()

`club_member_status` reaparece en el esquema aunque no esta dentro de los archivos Parquet fisicos — Spark lo reconstruye a partir del nombre de la carpeta.

Filtra por la columna particionada y revisa el plan — deberias ver `PartitionFilters`, no solo `PushedFilters` (el que ya viste en S2, 3.6):

In [42]:
df_verificacion.filter(col("club_member_status") == "ACTIVE").explain(True)

== Parsed Logical Plan ==
'Filter '`=`('club_member_status, ACTIVE)
+- Relation [customer_id#1632,FN#1633,Active#1634,fashion_news_frequency#1635,age#1636,postal_code#1637,club_member_status#1638] parquet

== Analyzed Logical Plan ==
customer_id: string, FN: double, Active: double, fashion_news_frequency: string, age: int, postal_code: string, club_member_status: string
Filter (club_member_status#1638 = ACTIVE)
+- Relation [customer_id#1632,FN#1633,Active#1634,fashion_news_frequency#1635,age#1636,postal_code#1637,club_member_status#1638] parquet

== Optimized Logical Plan ==
Filter (isnotnull(club_member_status#1638) AND (club_member_status#1638 = ACTIVE))
+- Relation [customer_id#1632,FN#1633,Active#1634,fashion_news_frequency#1635,age#1636,postal_code#1637,club_member_status#1638] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [customer_id#1632,FN#1633,Active#1634,fashion_news_frequency#1635,age#1636,postal_code#1637,club_member_status#1638] Batched: true, DataFi

Para ver cuantas filas quedaron guardadas en cada particion (cada carpeta `club_member_status=...`), sin salir de Spark ni contar archivos a mano:


In [43]:
df_verificacion.groupBy("club_member_status").count().orderBy("club_member_status").show(truncate=False)


+------------------+-------+
|club_member_status|count  |
+------------------+-------+
|ACTIVE            |1272491|
|LEFT CLUB         |467    |
|PRE-CREATE        |92960  |
|UNKNOWN           |6062   |
+------------------+-------+



Ya terminaste de reutilizar `df_customers_valido` — libera la memoria que ocupaba cacheado:

In [44]:
df_customers_valido.unpersist()

DataFrame[customer_id: string, FN: double, Active: double, club_member_status: string, fashion_news_frequency: string, age: int, postal_code: string]

## 3.12 Documentar hallazgos y responder preguntas de reflexion

**Producto del paso:** notebook documentado con celdas markdown explicando cada resultado.

Agrega celdas markdown breves debajo de cada bloque de codigo explicando que hiciste y que observaste — es la base directa de la evidencia tecnica para 4.3.1.

**Reflexion tecnica breve** (5 a 8 lineas): ¿que diferencia encontraste entre `dropDuplicates()` y `Window`+`row_number()` al aplicarlos sobre `articles.csv`? ¿que columnas rellenaste con `.na.fill()` y cuales no, y por que? ¿por que `.between(25, 35)` y `(col("age") > 25) & (col("age") < 35)` no dan el mismo resultado? ¿que diferencia notaste entre `PushedFilters` (S2) y `PartitionFilters` (S3) en el plan de ejecucion?

_(Responde aqui)_